<img src="AndrejKarpathy.jpg" alt="Andrej Karpathy" width="160">

这次分享内容一部分来自 Andrej Karpathy 的课程 [Neural Networks: Zero to Hero](https://www.youtube.com/watch?v=VMj-3S1tku0&list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ&index=1)，一部分来自来自大模型。

# 训练好的模型本质上是一个函数

一个训练好的大模型，本质上就是一个函数：给定输入，按一套已经定好的计算，得到输出。

$$
y = f(x;\ \theta)
$$

- $x$：输入。一个数、一句话、一张图，都可以是 $x$。
- $\theta$：参数，也就是权重和偏置。这些数决定函数的具体形状。
- $y$：输出。比如类别、下一个字，或者一个数值。

神经网络是把加法、乘法和激活函数（比如 tanh）嵌套、组合成一个函数 $f$。

函数由结构和参数组成。看一个具体的函数：

$$
f(x) = ax^2 + bx + c
$$

这是一个一元二次函数：只有一个输入 $x$，最高次是 $x^2$。这里讨论的就是函数的结构。它约定了这个函数做什么计算操作。

参数是里面可以改的数：$a$、$b$、$c$。取 $a = 3$、$b = -4$、$c = 5$，就得到

$$
f(x) = 3x^2 - 4x + 5
$$

结构没有变，换一组 $a, b, c$，曲线就不一样。



类似的，大模型也能分成结构和参数两部分。

结构就是模型的架构：用不用 Transformer，有多少层、每层多宽、有多少个注意力头，词怎么变成向量，注意力、前馈网络、残差怎么接。这些在训练之前就定好，训练过程中通常不动。

<img src="transformer_figure1.png" alt="Transformer 模型架构" width="420">

图 1：Transformer 的模型结构。引自 Vaswani et al., *Attention Is All You Need*, NIPS 2017, Figure 1.

参数就是每个计算层中的权重，大模型训练就是在调整参数。

# 大模型训练

训练要解决两件事：1、评判这组参数 “好不好”；2、判断每个参数该往哪个方向改进。

## 损失函数

模型先按当前的参数算出一个预测，再拿它和正确答案比。比完得到一个数，这个数叫损失。损失越大，说明这组参数越差。训练的目标就是把它降下去。

这个损失，目前的大模型主流是用交叉熵来计算。模型要预测的是下一个词：它给词表里每个词一个概率，记作 $q$；正确答案写成分布 $p$，正确词上是 1，其余词上是 0。交叉熵是

$$
H(p, q) = -\sum_i p_i \log q_i
$$

$p$ 只有正确词那一项是 1，公式就收成

$$
L = -\log q_{\text{正确词}}
$$

假设正确答案是「猫」。模型分给「猫」的概率越接近 1，损失越接近 0；这个概率越小，损失越大。

预测一个数值时，常用的是平方误差。前面的二次函数如果希望 $f(2)$ 接近 3，现在算出 10，损失就是 $(10 - 3)^2 = 49$。大模型这里比的是下一个词的概率准不准，所以用上面的交叉熵。

损失也是一个函数。输入是全部参数 $\theta$，输出是一个数。

## 参数怎么改

只知道损失大，还不知道每个参数该往哪边挪。损失对某一个参数的导数，就是这个方向，也叫梯度。

- 梯度为正：这个参数再大一点，损失会升高，所以把它减小一点。
- 梯度为负：这个参数再大一点，损失会降低，所以把它增大一点。

每次挪多远，由学习率决定。学习率太大，会跨过损失较低的位置；太小，则降得很慢。更新写成

$$
\theta \leftarrow \theta - \text{学习率} \times \text{梯度}
$$

二次函数只有 $a$、$b$、$c$ 三个参数，三个都按这个式子改。大模型有几千亿个参数，就对每一个都做同一步。

## 一遍遍重复

一次不把全部数据都看完。拿一小批样本算平均损失和梯度，更新一次参数，再换下一批。这一小批叫一个 batch。

结构不动：有多少层、注意力怎么接，训练过程中保持原样。变的是每一层里面的权重。数据看得越多，这些权重就被推得越接近「下一个词的概率比较准」的那一组。
